In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import torch.nn as nn
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score
import seaborn as sns
import sys
import os
import gc

from text_pipeline import load_and_prepare_data

"""
Downsizing logic for this file:
base --> train encoder
small --> freeze encoder
tiny --> freeze + bottleneck head
"""

def run_distilbert(freeze_encoder: bool, mode: int, size="base"):
  train_dataset, val_dataset, test_dataset, label2id, id2label, class_weights = load_and_prepare_data(mode, 'distilbert-base-uncased', max_length=128)

  tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

  collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, return_tensors="pt")

  model = DistilBertForSequenceClassification.from_pretrained(
      'distilbert-base-uncased',
      num_labels=len(label2id),
      id2label=id2label,
      label2id=label2id
  )

  if freeze_encoder:
    for param in model.distilbert.parameters():
      param.requires_grad = False

    if size == "tiny":
      hidden_size = model.distilbert.config.dim
      model.pre_classifier = nn.Sequential(
          nn.Linear(hidden_size, 128),
          nn.ReLU(),
          nn.Dropout(0.1),
      )
      model.classifier = nn.Linear(128, len(label2id))

  class WeightedTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
      labels = inputs.get("labels")

      outputs = model(**inputs)
      logits = outputs.get("logits")

      loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))

      loss = loss_fct(
          logits.view(-1, self.model.config.num_labels),
          labels.view(-1)
      )

      return (loss, outputs) if return_outputs else loss


  def compute_metrics(eval_pred):
      logits, labels = eval_pred
      predictions = np.argmax(logits, axis=-1)

      macro_f1 = f1_score(
          labels,
          predictions,
          average='macro'
      )

      accuracy = accuracy_score(labels, predictions)

      return {
          'accuracy': accuracy,
          'macro_f1': macro_f1
      }

  training_args = TrainingArguments(
      output_dir='./distilbert-results',
      eval_strategy='epoch',
      save_strategy='epoch',
      learning_rate=2e-5,
      num_train_epochs=3,
      per_device_train_batch_size=8,
      per_device_eval_batch_size=8,
      logging_steps=100,
      weight_decay=0.01,
      load_best_model_at_end=True,
      metric_for_best_model='macro_f1',
      greater_is_better=True
  )

  trainer = WeightedTrainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=val_dataset,
      compute_metrics=compute_metrics,
      processing_class=tokenizer,
      data_collator=collator
  )

  trainer.train()

  metrics = trainer.evaluate(
      eval_dataset=test_dataset
  )

  history = trainer.state.log_history

  train_losses = []
  val_losses = []

  preds = trainer.predict(test_dataset)

  y_pred = np.argmax(preds.predictions, axis=-1)
  y_true = preds.label_ids

  for entry in history:
    if "loss" in entry and "epoch" in entry:
      train_losses.append(entry["loss"])

    if "eval_loss" in entry:
      val_losses.append(entry["eval_loss"])

  accuracy = accuracy_score(y_true, y_pred)
  precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
  cm = confusion_matrix(y_true, y_pred)

  del model
  del trainer
  del tokenizer
  del collator
  gc.collect()
  torch.cuda.empty_cache()


  return {
      'accuracy': accuracy,
      'precision': precision,
      'recall': recall,
      'f1': f1,
      'train_loss': train_losses,
      'val_loss': val_losses,
      'confusion_matrix': cm,
      'history': val_losses
  }

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'eval_loss': 0.019678710028529167, 'eval_macro_f1': 0.984921392578638, 'eval_runtime': 57.4709, 'eval_samples_per_second': 550.836, 'eval_steps_per_second': 68.87, 'epoch': 3.0}
